In [1]:
import re
import random
import pandas as pd
from collections import Counter

In [2]:
RESUME_FILE = "Resume.csv"
QUESTION_FILE = "train-00000-of-00001.parquet"   # the Hugging Face file
JOBS_FILE = "postings.csv"

In [10]:
USE_MODEL = True
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [5]:
skILLS = ["python", "java", "javascript", "c++", "sql", "html", "css", "react",
          "django", "flask", "git", "docker", "aws", "linux", "api",
          "machine learning", "deep learning", "pandas", "numpy", "tensorflow",
          "data structures", "algorithms", "oop", "testing", "statistics",
          "excel", "tableau", "networking", "cloud"]

BEHAVIORAL_WORDS = ["tell me about", "describe a time", "how do you handle",
                    "strength", "weakness", "team", "conflict", "deadline",
                    "mistake", "challenge", "feedback", "leadership"]

DEFAULT_BEHAVIORAL = [
    "Tell me about a time you had to learn something new quickly.",
    "Describe a disagreement with a teammate and how you solved it.",
    "Tell me about a project you are proud of.",
    "How do you manage your time when you have many deadlines?",
    "Describe a mistake you made and what you learned from it.",
]


def find_skills(text):
    """Return the skills from SKILLS that appear in the text."""
    text = str(text).lower()
    found = []
    for skill in SKILLS:
        pattern = r"(?<![a-z0-9+#])" + re.escape(skill) + r"(?![a-z0-9+#])"
        if re.search(pattern, text):
            found.append(skill)
    return found

In [8]:
import csv

# Fix: Load resumes while handling unclosed quotes or malformed lines using the python engine and on_bad_lines='skip'
resumes = pd.read_csv(RESUME_FILE, engine="python", on_bad_lines="skip")
resumes = resumes[resumes["Category"].isin(["INFORMATION-TECHNOLOGY", "ENGINEERING"])]

# Fix: Standardize reference to "skILLS" by mapping SKILLS to skILLS in the current namespace
SKILLS = skILLS
resumes["skills"] = resumes["Resume_str"].apply(find_skills)

# Question bank: parquet file. We look for the column that holds the questions.
bank = pd.read_parquet(QUESTION_FILE) if QUESTION_FILE.endswith(".parquet") else pd.read_csv(QUESTION_FILE)
print("Question file columns:", list(bank.columns))
q_cols = [c for c in bank.columns if "question" in c.lower()]
QUESTION_COLUMN = q_cols[0] if q_cols else bank.columns[0]   # change this if it picks the wrong column
print("Using column:", QUESTION_COLUMN)

bank = bank[[QUESTION_COLUMN]].dropna().drop_duplicates()
bank.columns = ["question"]
bank["skills"] = bank["question"].apply(find_skills)
bank["is_behavioral"] = bank["question"].str.lower().apply(
    lambda q: any(w in q for w in BEHAVIORAL_WORDS))

# Job postings: used to learn which skills a role needs
jobs = pd.read_csv(JOBS_FILE, on_bad_lines='skip')
print("Job file columns:", list(jobs.columns))
title_col = "job_title" if "job_title" in jobs.columns else "title"
text_col = next(c for c in ["job_skills", "skills_desc", "job_summary", "description"] if c in jobs.columns)

Question file columns: ['Question', 'Answer', 'Correctness', 'FollowUp Question']
Using column: Question
Job file columns: ['job_title', 'company', 'job_location', 'job_link', 'first_seen', 'search_city', 'search_country', 'job level', 'job_type', 'job_summary', 'job_skills']


In [11]:
generator = None
if USE_MODEL:
    from transformers import pipeline
    generator = pipeline("text-generation", model=MODEL_NAME)


def generate_new_questions(role, skills, examples, qtype, n):
    """Ask the LLaMA model to write new questions, using bank questions as examples."""
    if generator is None or not examples:
        return []
    shots = "\n".join("- " + q for q in examples[:3])
    prompt = (f"Write ONE new {qtype} interview question for a {role} intern.\n"
              f"Skills: {', '.join(skills) if skills else 'general programming'}.\n"
              f"Example questions:\n{shots}\n"
              f"Reply with only the new question.")
    messages = [{"role": "system", "content": "You are an expert technical interviewer."},
                {"role": "user", "content": prompt}]
    chat_prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False,
                                                          add_generation_prompt=True)
    new = []
    for _ in range(n * 3):                      # a few tries, some outputs get thrown away
        if len(new) >= n:
            break
        out = generator(chat_prompt, max_new_tokens=60, do_sample=True, temperature=0.8,
                        top_p=0.95, return_full_text=False)
        text = out[0]["generated_text"].strip().split("\n")[0].strip(' "-*0123456789.')
        if len(text) > 15 and text.endswith("?") and text not in examples and text not in new:
            new.append(text)
    return new

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [12]:
def skills_for_role(role, top=5):
    """Most common skills in job postings whose title contains the role."""
    rows = jobs[jobs[title_col].astype(str).str.lower().str.contains(role.lower())].head(300)
    counter = Counter()
    for text in rows[text_col].fillna(""):
        counter.update(find_skills(text))
    return [skill for skill, _ in counter.most_common(top)]


def make_questions(skills, role="software", n_technical=5, n_behavioral=3):
    """Technical + behavioral questions: new ones from the model, topped up from the bank."""
    technical = bank[~bank["is_behavioral"]]
    matching = technical[technical["skills"].apply(lambda s: any(x in skills for x in s))]
    pool = matching["question"].tolist() or technical["question"].tolist()
    tech_examples = random.sample(pool, min(n_technical, len(pool)))

    behavioral = bank[bank["is_behavioral"]]["question"].tolist() + DEFAULT_BEHAVIORAL
    beh_examples = random.sample(behavioral, min(n_behavioral, len(behavioral)))

    tech = generate_new_questions(role, skills, tech_examples, "technical", n_technical // 2 + 1)
    beh = generate_new_questions(role, skills, beh_examples, "behavioral", n_behavioral // 2)
    tech = (["[AI] " + q for q in tech] + tech_examples)[:n_technical]
    beh = (["[AI] " + q for q in beh] + beh_examples)[:n_behavioral]
    return tech, beh


def show(title, skills, tech, beh):
    print("\n" + "=" * 60)
    print("Candidate / role:", title)
    print("Skills:", ", ".join(skills) if skills else "none found")
    print("\nTechnical questions:")
    for i, q in enumerate(tech, 1):
        print(f"  {i}. {q}")
    print("\nBehavioral questions:")
    for i, q in enumerate(beh, 1):
        print(f"  {i}. {q}")


In [13]:
random.seed(42)
results = []

# A) Questions for 3 interns, based on their resumes
for _, row in resumes.head(3).iterrows():
    tech, beh = make_questions(row["skills"], role=row["Category"].replace("-", " ").lower())
    show("Resume ID " + str(row["ID"]), row["skills"], tech, beh)
    results.append({"who": "resume " + str(row["ID"]), "skills": ", ".join(row["skills"]),
                    "technical": " | ".join(tech), "behavioral": " | ".join(beh)})

# B) Questions for a role, based on the job postings
for role in ["python developer", "data scientist"]:
    skills = skills_for_role(role)
    tech, beh = make_questions(skills, role=role)
    show("Role: " + role, skills, tech, beh)
    results.append({"who": role, "skills": ", ".join(skills),
                    "technical": " | ".join(tech), "behavioral": " | ".join(beh)})

pd.DataFrame(results).to_csv("generated_questions.csv", index=False)
print("\nSaved to generated_questions.csv")


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Bot


Candidate / role: Resume ID 36856210
Skills: html, linux

Technical questions:
  1. [AI] Question: How does the use of eventual consistency in database management systems ensure immediate consistency, and what are the limitations of this approach?
  2. [AI] Can you explain the significance of event sourcing in CQRS and how it differs from other event-based architecture patterns?
  3. [AI] New Technical Interview Question: What is the difference between eventual consistency and immediate consistency in a system that uses eventual consistency?
  4. What is the significance of event sourcing in CQRS?
  5. What is the purpose of model deployment in machine learning?

Behavioral questions:
  1. [AI] QUESTION: What is the best way to ensure a website's security and prevent security breaches?
  2. What are some challenges of implementing real-time systems?
  3. What is the key challenge of symmetric encryption?


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Candidate / role: Resume ID 21780877
Skills: testing, networking

Technical questions:
  1. [AI] Question: In your experience, what are the most effective ways to test for vulnerabilities in a software application?
  2. [AI] Question: What is the role of security testing in software development and testing?
  3. [AI] Question: What is the significance of performance testing in software development?
  4. What is the primary purpose of load testing in software engineering?
  5. What types of tests are typically included in a DevOps automated testing pipeline?

Behavioral questions:
  1. [AI] QUESTION: In a team project, when should you consider refactoring code to improve its overall quality?
  2. When should you consider refactoring code in a team project?
  3. What are the main challenges faced by Secure Multi-Party Computation?


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Candidate / role: Resume ID 33241454
Skills: excel

Technical questions:
  1. [AI] Question: How does the use of Differential Privacy benefit the security of the system during penetration testing?
  2. [AI] Question: What is the difference between a vulnerability scanner and a penetration tester in the context of penetration testing?
  3. [AI] How does threat modeling relate to the development and maintenance of an IT system?
  4. What is the role of a vulnerability scanner in penetration testing?
  5. What is the purpose of threat modeling in the SDLC?

Behavioral questions:
  1. [AI] Behavioral Interview Question: Describe a scenario where you had to prioritize multiple projects based on their urgency, deadline, and importance. How did you weigh the factors and make a decision?
  2. Describe a disagreement with a teammate and how you solved it.
  3. What is the role of a Product Owner in an Agile team?


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Candidate / role: Role: python developer
Skills: sql, linux

Technical questions:
  1. [AI] Question: Can you summarize the purpose of the LIMIT clause in SQL and provide an example of how it is used?
  2. [AI] Question: "What is the difference between a SELECT statement and an ORDER BY statement in SQL?
  3. [AI] Question: What is the difference between SELECT and FROM in SQL?
  4. What is the purpose of the LIMIT clause in SQL?
  5. What is the purpose of the SELECT statement in SQL?

Behavioral questions:
  1. [AI] Question: Describe how Python's built-in `sqlite3` library can be used to connect to a SQLite database and retrieve data from it. Specifically, what are the steps involved in setting up the database connection, executing queries, and retrieving data?
  2. What is the primary challenge of Quantum Cryptography?
  3. How can teams manage technical debt effectively?


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Candidate / role: Role: data scientist
Skills: none found

Technical questions:
  1. [AI] What is Apache Kafka and how does it work?
  2. [AI] Question: What is the difference between Amazon Elastic Container Registry (ECR) and Docker Hub?
  3. [AI] What is Kubernetes used for and how is it different from Amazon EC2?
  4. What is Kubernetes used for?
  5. What is Amazon RDS and how is it different from EC2?

Behavioral questions:
  1. [AI] What is the key difference between a linear regression model and a logistic regression model?
  2. What challenges are typically encountered when using RL for software optimization?
  3. What is the key challenge of symmetric encryption?

Saved to generated_questions.csv
